In [47]:
from netgen.occ import *
from netgen.meshing import IdentificationType
from ngsolve import *
from ngsolve.webgui import Draw

# 1. Geometry Setup
L = 2.0
pml_thickness = 0.8
z_source = 1.0  # Height of our invisible source plane

# Stack the boxes from bottom to top and name their materials
# 1. PML 
box_pml_d = Box((0, 0, -pml_thickness), (L/2, L/2, 0))
box_pml_d.mat("pml_region_down")  # <-- Added material name

box_pml_u = Box((0, 0, L), (L/2, L/2, L+pml_thickness))
box_pml_u.mat("pml_region_up")  # <-- Added material name

# 2. Lower Air (Between PML and Source)
box_air_bottom = Box((0, 0, 0), (L/2, L/2, z_source))
box_air_bottom.mat("air")  # <-- Added material name

# 3. Upper Air (Between Source and Rigid Wall)
box_air_top = Box((0, 0, z_source), (L/2, L/2, L))
box_air_top.mat("air")     # <-- Added material name

# Glue them into a single continuous domain
geometry = Glue([box_pml_d, box_air_bottom, box_air_top, box_pml_u])

# 2. Name the Faces
# Netgen's OCC allows us to gracefully select faces by their coordinate planes
geometry.faces[Z == z_source].name = "source_plane"

# 3. Identify Periodic Faces (on the outer boundaries of the glued geometry)
geometry.faces[X == L/2].Identify(geometry.faces[X == 0], "periodic_x", IdentificationType.PERIODIC)
geometry.faces[Y == L/2].Identify(geometry.faces[Y == 0], "periodic_y", IdentificationType.PERIODIC)

# 4. Generate Mesh
geo = OCCGeometry(geometry)
mesh = Mesh(geo.GenerateMesh(maxh=0.2))

print("Mesh generated successfully! Notice the internal face at z=0.2")
Draw(mesh)

Mesh generated successfully! Notice the internal face at z=0.2


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [ ]:
import cmath, math
# 1. Physics Parameters
freq = 343.0*2         
c_0 = 343.0          
k = 2 * math.pi * freq / c_0  

# Incident angles (e.g., 45 degrees)
theta = math.pi / 4 
phi = 0.0

# Wave vector components
kx = k * math.sin(theta) * math.cos(phi)
ky = k * math.sin(theta) * math.sin(phi)
kz = k * math.cos(theta)

# Transverse wave vector for the Floquet shift
k_vec = CF((kx, ky, 0)) 

# 2. Finite Element Space
fes = H1(mesh, order=3, complex=True)
u, v = fes.TnT()

print(f"Degrees of freedom: {fes.ndof}")


Degrees of freedom: 13263


In [49]:
from ngsolve import pml

# 1. Activate the PML, specifically targeting the "pml_region" material
mesh.SetPML(pml.HalfSpace(point=(0,0,0), normal=(0,0,-1), alpha=1j), "pml_region_down") # <-- Added "pml_region"
mesh.SetPML(pml.HalfSpace(point=(0,0,L), normal=(0,0,1), alpha=1j), "pml_region_up") # <-- Added "pml_region"

# 2. Modified Gradients (Floquet Trick)
grad_u = grad(u) + 1j * k_vec * u
grad_v = grad(v) - 1j * k_vec * v 

# 3. Bilinear Form (LHS)
a = BilinearForm(fes)
a += (grad_u * grad_v - k**2 * u * v) * dx

# 4. Linear Form (RHS) - The Transparent Source
f = LinearForm(fes)
source_func = exp(-40 * (z-z_source)**2)
f += source_func* v * dx

# 5. Assemble and Solve
gfu = GridFunction(fes)

with TaskManager():
    a.Assemble()
    f.Assemble()
    gfu.vec.data = a.mat.Inverse() * f.vec
    
print("Solve complete!")


Solve complete!


In [51]:
# 1. Reconstruct Field
# We only need to multiply our envelope by the transverse Floquet phase
phase = exp(1j * (kx * x + ky * y))

# This is the true, physical Total Acoustic Pressure
p_true = gfu * phase

# 2. Visualize
print("Rendering total pressure field...")
# Use animate_complex=True and turn on Clipping in the WebGUI settings!
Draw(p_true, mesh, "Acoustic Pressure", animate_complex=True)

Rendering total pressure field...


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

BaseWebGuiScene